<!-- SPDX-License-Identifier: Apache-2.0 -->
# PySpark in JupyterLite — demo

Runs the **real** PySpark Connect client in your browser tab (Pyodide), talking to a Spark Connect server through a grpc-web/`fetch` transport. Calls are made blocking via an `Atomics`/`SharedArrayBuffer` bridge, so `.collect()` / `.toPandas()` work synchronously.

Requires a cross-origin-isolated page (COOP/COEP) and a running Spark Connect server behind an Envoy grpc-web proxy.

## 0. Assert cross-origin isolation
SharedArrayBuffer / Atomics.wait only exist on a cross-origin-isolated page. Fail loud and early.

In [ ]:
import js
assert getattr(js, 'crossOriginIsolated', False), (
    'Page is NOT cross-origin isolated. Serve with '
    'COOP: same-origin and COEP: credentialless.'
)
print('crossOriginIsolated =', js.crossOriginIsolated)

## 1. Install the monkey-patch
`pcw.install()` is idempotent; it swaps PySpark's gRPC stub for the grpc-web transport (the components) backed by the SAB bridge (the components).

In [ ]:
import js, micropip
origin = js.location.origin
await micropip.install('protobuf>=7')
await micropip.install('googleapis-common-protos>=1.56.4')
await micropip.install('zstandard')
# Slim Spark Connect client; deps=False because its grpcio/grpcio-status base
# deps have no Pyodide wheel and are stubbed by pyspark-connect-web's shim.
await micropip.install(f'{origin}/pyspark_client-4.1.2-py3-none-any.whl', deps=False)
await micropip.install(f'{origin}/pyspark_connect_web-0.2.0-py3-none-any.whl')

import pyspark_connect_web as pcw
pcw.install()

## 2. Open a session and run the read path
`sc://<envoy-host>:<port>/;transport=grpcweb` (the transport contract §2). Everything below is unchanged PySpark.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .remote('sc://localhost:8081/;transport=grpcweb')
    .getOrCreate()
)

# Blocking .toPandas() — the Atomics/SAB bridge makes this synchronous.
spark.range(10).toPandas()

In [ ]:
# A fuller read path.
from pyspark.sql import functions as F
(
    spark.range(100)
    .filter('id % 2 = 0')
    .select((F.col('id') % 3).alias('g'), F.col('id'))
    .groupBy('g')
    .agg(F.count('*').alias('n'))
    .toPandas()
)

In [ ]:
spark.sql('select 1 as x').collect()